In [1]:
import json
import shutil
from pathlib import Path
from collections import defaultdict

In [2]:
# Anchor paths from notebook
BASE_DIR = Path().cwd().parents[2]
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed_data"
SAVE_DIR = BASE_DIR / "data"  / "total"
SAVE_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Track all furniture globally
for room_dir in PROCESSED_DATA_DIR.iterdir():
    if not room_dir.is_dir():
        continue
        
    room_type = room_dir.name
    print(f"Room {room_type}")
    
    # Initialize tracking variables per room type
    general_manifest = []
    category_data = defaultdict(list)  # category -> list of {furniture_id, image_name}
    seen_furniture_ids = set()  # Track seen IDs to prevent duplicates
    
    # Directory to save organized data for this room type
    room_save_dir = SAVE_DIR / room_type
    room_save_dir.mkdir(parents=True, exist_ok=True)

#  "wayfair"
    # Process each source within the room type
    for source_dir in room_dir.glob("*/"):
        if source_dir.name in ["deepfurn", "sklad_mebliv"]:
            print(f"  Source: {source_dir.name}...")
            
            # Iterate through scene folders
            for scene_folder in sorted(source_dir.glob("*_[0-9][0-9][0-9][0-9]")):
                if not scene_folder.is_dir():
                    continue
                
                # Read annotation.json
                annotation_path = scene_folder / "annotations.json"
                if not annotation_path.exists():
                    continue
                
                with open(annotation_path, "r", encoding="utf-8") as f:
                    annotation = json.load(f)
                    
                # Process each furniture item
                furnitures_dir = scene_folder / "furnitures"
                if furnitures_dir.exists():
                    for furniture in annotation.get("furnitures", []):
                        furniture_id = furniture.get("furniture_id")
                        category = furniture.get("category", "unknown").lower()
                        
                        # Skip if furniture_id already processed
                        if furniture_id in seen_furniture_ids:
                            continue
                        
                        # Find corresponding furniture image
                        image_name = f"{furniture_id}.jpg"
                        furniture_image_path = furnitures_dir / image_name
                        
                        if furniture_image_path.exists():
                            # Create category folder within the room type directory
                            category_folder = room_save_dir / category
                            category_folder.mkdir(parents=True, exist_ok=True)
                            
                            # Copy image to category folder
                            dest_image_path = category_folder / image_name
                            shutil.copy2(furniture_image_path, dest_image_path)
                            
                            # Track for manifests
                            record = {
                                "furniture_id": furniture_id,
                                "image_name": image_name,
                                "source": source_dir.name,
                                "scene": scene_folder.name
                            }
                            
                            category_data[category].append(record)
                            general_manifest.append({**record, "category": category})
                            seen_furniture_ids.add(furniture_id)

    # Save manifests per room type
    for category, records in category_data.items():
        category_manifest = {
            "category": category,
            "furniture_count": len(records),
            "furnitures": records
        }
        category_manifest_path = room_save_dir / category / "category_manifest.json"
        with open(category_manifest_path, "w", encoding="utf-8") as f:
            json.dump(category_manifest, f, indent=2, ensure_ascii=False)
        print(f"    {category} - {len(records)} items")

    # Write general manifest for the room type
    general_manifest_path = room_save_dir / "general_manifest.json"
    general_summary = {
        "room_type": room_type,
        "total_furniture_items": len(general_manifest),
        "unique_categories": len(category_data),
        "categories": list(category_data.keys()),
        "furnitures": general_manifest
    }
    with open(general_manifest_path, "w", encoding="utf-8") as f:
        json.dump(general_summary, f, indent=2, ensure_ascii=False)

    print(f"  Total furniture items for {room_type}: {len(seen_furniture_ids)}")
    print(f"  Unique categories for {room_type}: {len(category_data)}\n")

Room bedrooms
  Source: deepfurn...
  Source: sklad_mebliv...
    chair_stool - 357 items
    table - 317 items
    curtain - 431 items
    bed - 553 items
    small_storage - 636 items
    large_storage - 368 items
  Total furniture items for bedrooms: 2662
  Unique categories for bedrooms: 6

Room living_rooms
  Source: deepfurn...
    curtain - 222 items
    table - 627 items
    sofa - 822 items
    small_storage - 324 items
    chair_stool - 220 items
    large_storage - 294 items
  Total furniture items for living_rooms: 2509
  Unique categories for living_rooms: 6

